# Pipeline Medallion — ANS Beneficiários SP

Versão notebook (estilo Databricks) do pipeline. Executa **Bronze → Silver → Gold** e as **3 consultas** do case.

> O mesmo código roda no Databricks: basta apontar `LAKE_ROOT` para o S3 real. Aqui usamos MinIO (S3 local) via Docker.

In [1]:
import sys
sys.path.append('/app')

from src.config import get_spark, sql_params
from src.sql_runner import run_sql_file
from src.queries import run_queries

spark = get_spark('case-ans-notebook')
params = sql_params()
params

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f653950f-f0a1-4d4b-9753-d1e2b8c740ce;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 206ms :: artifacts dl 10ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 

{'LAKE_ROOT': 's3a://lakehouse',
 'CSV_PATH': '/app/data/pda-024-icb-SP-2025_08.csv'}

## Bronze — ingestão as-is
Carrega o CSV bruto para Delta, preservando a estrutura original.

In [2]:
run_sql_file(spark, '00_bronze.sql', params)
spark.table('bronze.beneficiarios').limit(5).toPandas()


=== 00_bronze.sql  (3 statements) ===
  [1/3] CREATE DATABASE IF NOT EXISTS bronze LOCATION 's3a://lakehouse/bronze'...


26/08/02 05:24:19 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/08/02 05:24:19 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/08/02 05:25:03 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/08/02 05:25:03 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@172.30.0.8
26/08/02 05:25:03 WARN ObjectStore: Failed to get database default, returning NoSuchObjectException
26/08/02 05:25:07 WARN ObjectStore: Failed to get database bronze, returning NoSuchObjectException
26/08/02 05:25:07 WARN ObjectStore: Failed to get database bronze, returning NoSuchObjectException
26/08/02 05:25:07 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/08/02 05:25:07 WARN ObjectStore: Failed to get database bronze, returning NoSuchOb

  [2/3] CREATE OR REPLACE TEMPORARY VIEW raw_csv USING csv OPTIONS ( path '/app/data/pda...
  [3/3] CREATE OR REPLACE TABLE bronze.beneficiarios USING delta AS SELECT *, current_ti...


26/08/02 05:25:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/08/02 05:25:32 WARN OptimisticTransaction: [tableId=af61115d,txnId=29cee5a6] Change in the table id detected in txn. Table id for txn on table at s3a://lakehouse/bronze/beneficiarios was af61115d-4d6b-434a-81ee-53c23b9d2efc when the txn was created and is now changed to 55956880-d82c-4475-a963-62764dbaa9cd.
26/08/02 05:25:33 WARN DeltaLog: Change in the table id detected while updating snapshot. 
Previous snapshot = Snapshot(path=s3a://lakehouse/bronze/beneficiarios/_delta_log, version=4, metadata=Metadata(af61115d-4d6b-434a-81ee-53c23b9d2efc,null,null,Format(parquet,Map()),{"type":"struct","fields":[{"name":"ID_CMPT_MOVEL","type":"string","nullable":true,"metadata":{}},{"name":"CD_OPERADORA","type":"string","nullable":true,"metadata":{}},{"name":"NM_RAZAO_SOCIAL","type":"string","nullable":true,"

,ID_CMPT_MOVEL,CD_OPERADORA,NM_RAZAO_SOCIAL,NR_CNPJ,MODALIDADE_OPERADORA,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,TP_SEXO,DE_FAIXA_ETARIA,...,DE_SEGMENTACAO_PLANO,DE_ABRG_GEOGRAFICA_PLANO,COBERTURA_ASSIST_PLAN,TIPO_VINCULO,QT_BENEFICIARIO_ATIVO,QT_BENEFICIARIO_ADERIDO,QT_BENEFICIARIO_CANCELADO,DT_CARGA,_ingested_at,_source_file
0,2025-08,333051,UNIMED DE GUARULHOS COOPERATIVA DE TRABALHO MÉ...,74466137000172,COOPERATIVA MÉDICA,SP,351880,Guarulhos,F,20 a 24 anos,...,Ambulatorial + Hospitalar com obstetrícia,Municipal,Médico-hospitalar,Titular,1,0,0,2026-06-30,2026-08-02 05:25:11.458589,/app/data/pda-024-icb-SP-2025_08.csv
1,2025-08,350249,H.B. SAÚDE S/A.,02668512000156,MEDICINA DE GRUPO,SP,351120,Catiguá,M,30 a 34 anos,...,Ambulatorial + Hospitalar com obstetrícia,Grupo de municípios,Médico-hospitalar,Titular,1,0,0,2026-06-30,2026-08-02 05:25:11.458589,/app/data/pda-024-icb-SP-2025_08.csv
2,2025-08,364312,UNIMED DE ARARAQUARA - COOP. DE TRAB. MÉDICO,45272366000158,COOPERATIVA MÉDICA,SP,350320,Araraquara,F,65 a 69 anos,...,Ambulatorial + Hospitalar com obstetrícia,Grupo de municípios,Médico-hospitalar,Titular,1,0,0,2026-06-30,2026-08-02 05:25:11.458589,/app/data/pda-024-icb-SP-2025_08.csv
3,2025-08,346659,CAIXA DE ASSISTÊNCIA DOS FUNCIONÁRIOS DO BANCO...,33719485000127,AUTOGESTÃO,SP,354980,São José do Rio Preto,M,50 a 54 anos,...,Ambulatorial + Hospitalar com obstetrícia,Nacional,Médico-hospitalar,Titular,3,0,0,2026-06-30,2026-08-02 05:25:11.458589,/app/data/pda-024-icb-SP-2025_08.csv
4,2025-08,379956,CARE PLUS MEDICINA ASSISTENCIAL LTDA.,02725347000127,MEDICINA DE GRUPO,SP,352400,Itupeva,F,30 a 34 anos,...,Ambulatorial + Hospitalar com obstetrícia,Grupo de estados,Médico-hospitalar,Titular,2,0,0,2026-06-30,2026-08-02 05:25:11.458589,/app/data/pda-024-icb-SP-2025_08.csv


## Silver — tipagem + mascaramento
Tipa as colunas, mascara o CNPJ e particiona por competência.

In [3]:
run_sql_file(spark, '01_silver.sql', params)
spark.table('silver.beneficiarios').limit(5).toPandas()


=== 01_silver.sql  (2 statements) ===
  [1/2] CREATE DATABASE IF NOT EXISTS silver LOCATION 's3a://lakehouse/silver'...
  [2/2] CREATE OR REPLACE TABLE silver.beneficiarios USING delta PARTITIONED BY (id_cmpt...


26/08/02 05:26:05 WARN ObjectStore: Failed to get database silver, returning NoSuchObjectException
26/08/02 05:26:05 WARN ObjectStore: Failed to get database silver, returning NoSuchObjectException
26/08/02 05:26:05 WARN ObjectStore: Failed to get database silver, returning NoSuchObjectException
26/08/02 05:26:15 WARN OptimisticTransaction: [tableId=fce75241,txnId=3665729f] Change in the table id detected in txn. Table id for txn on table at s3a://lakehouse/silver/beneficiarios was fce75241-21d2-41e1-99eb-fca18c3d9366 when the txn was created and is now changed to d7fe35cd-5a89-4b7c-88a6-c993b707b1d8.
26/08/02 05:26:16 WARN DeltaLog: Change in the table id detected while updating snapshot. 
Previous snapshot = Snapshot(path=s3a://lakehouse/silver/beneficiarios/_delta_log, version=4, metadata=Metadata(fce75241-21d2-41e1-99eb-fca18c3d9366,null,null,Format(parquet,Map()),{"type":"struct","fields":[{"name":"cd_operadora","type":"string","nullable":true,"metadata":{}},{"name":"nm_razao_soci

,cd_operadora,nm_razao_social,nr_cnpj_masc,modalidade_operadora,sg_uf,cd_municipio,nm_municipio,tp_sexo,de_faixa_etaria,de_segmentacao_plano,de_abrg_geografica_plano,de_contratacao_plano,cobertura_assist_plan,tipo_vinculo,qt_beneficiario_ativo,qt_beneficiario_aderido,qt_beneficiario_cancelado,dt_carga,id_cmpt_movel
0,000582,PORTO SEGURO - SEGURO SAÚDE S/A,04540010******,SEGURADORA ESPECIALIZADA EM SAÚDE,SP,355100,São Vicente,M,20 a 24 anos,Odontológico,Nacional,Coletivo Empresarial,Odontológico,Titular,14,0,1,2026-06-30,2025-08
1,355721,UNIMED DE SANTOS COOP DE TRAB MEDICO,58229691******,COOPERATIVA MÉDICA,SP,354100,Praia Grande,F,65 a 69 anos,Ambulatorial + Hospitalar com obstetrícia,Grupo de municípios,Individual ou Familiar,Médico-hospitalar,Dependente,1,0,0,2026-06-30,2025-08
2,312126,FUNDAÇÃO SAÚDE ITAÚ,73809352******,AUTOGESTÃO,SP,350900,Caieiras,M,35 a 39 anos,Ambulatorial + Hospitalar com obstetrícia,Nacional,Coletivo Empresarial,Médico-hospitalar,Titular,1,0,0,2026-06-30,2025-08
3,419419,BRASILDENTAL OPERADORA DE PLANOS ODONTOLÓGICOS...,19962272******,ODONTOLOGIA DE GRUPO,SP,350600,Bauru,M,35 a 39 anos,Odontológico,Nacional,Coletivo Empresarial,Odontológico,Titular,10,0,0,2026-06-30,2025-08
4,312924,CAIXA ECONÔMICA FEDERAL,00360305******,AUTOGESTÃO,SP,350370,Ariranha,F,40 a 44 anos,Ambulatorial + Hospitalar com obstetrícia + Od...,Nacional,Coletivo por Adesão,Médico-hospitalar,Titular,1,0,0,2026-06-30,2025-08


## Gold — tabelas curadas
Agrega os dados para consumo analítico.

In [4]:
run_sql_file(spark, '02_gold.sql', params)
spark.sql('SHOW TABLES IN gold').toPandas()


=== 02_gold.sql  (4 statements) ===
  [1/4] CREATE DATABASE IF NOT EXISTS gold LOCATION 's3a://lakehouse/gold'...
  [2/4] CREATE OR REPLACE TABLE gold.beneficiarios_por_operadora USING delta AS SELECT c...


26/08/02 05:27:47 WARN ObjectStore: Failed to get database gold, returning NoSuchObjectException
26/08/02 05:27:47 WARN ObjectStore: Failed to get database gold, returning NoSuchObjectException
26/08/02 05:27:47 WARN ObjectStore: Failed to get database gold, returning NoSuchObjectException
26/08/02 05:27:48 WARN OptimisticTransaction: [tableId=92ca826c,txnId=76de63e1] Change in the table id detected in txn. Table id for txn on table at s3a://lakehouse/gold/beneficiarios_por_operadora was 92ca826c-67f0-40bb-8e54-2baec78ffebe when the txn was created and is now changed to 3ebccd36-950c-4eee-adc6-c85917636eed.
26/08/02 05:27:49 WARN DeltaLog: Change in the table id detected while updating snapshot. 
Previous snapshot = Snapshot(path=s3a://lakehouse/gold/beneficiarios_por_operadora/_delta_log, version=4, metadata=Metadata(92ca826c-67f0-40bb-8e54-2baec78ffebe,null,null,Format(parquet,Map()),{"type":"struct","fields":[{"name":"cd_operadora","type":"string","nullable":true,"metadata":{}},{"na

  [3/4] CREATE OR REPLACE TABLE gold.beneficiarios_por_faixa_etaria USING delta AS SELEC...


26/08/02 05:27:50 WARN OptimisticTransaction: [tableId=82c90346,txnId=bb9f7a7a] Change in the table id detected in txn. Table id for txn on table at s3a://lakehouse/gold/beneficiarios_por_faixa_etaria was 82c90346-bf38-4d7b-a47c-3295aedaeb46 when the txn was created and is now changed to c110cbae-3a3e-44bb-9e90-11d90c450c4a.
26/08/02 05:27:50 WARN DeltaLog: Change in the table id detected while updating snapshot. 
Previous snapshot = Snapshot(path=s3a://lakehouse/gold/beneficiarios_por_faixa_etaria/_delta_log, version=4, metadata=Metadata(82c90346-bf38-4d7b-a47c-3295aedaeb46,null,null,Format(parquet,Map()),{"type":"struct","fields":[{"name":"de_faixa_etaria","type":"string","nullable":true,"metadata":{}},{"name":"qt_beneficiarios_ativos","type":"long","nullable":true,"metadata":{}}]},List(),Map(),Some(1785647529118)), logSegment=LogSegment(s3a://lakehouse/gold/beneficiarios_por_faixa_etaria/_delta_log,4,WrappedArray(S3AFileStatus{path=s3a://lakehouse/gold/beneficiarios_por_faixa_etaria

  [4/4] CREATE OR REPLACE TABLE gold.beneficiarios_por_municipio USING delta AS SELECT c...


26/08/02 05:27:52 WARN OptimisticTransaction: [tableId=8a9eb749,txnId=c8aa9a22] Change in the table id detected in txn. Table id for txn on table at s3a://lakehouse/gold/beneficiarios_por_municipio was 8a9eb749-121b-4eb2-8a38-e359138990eb when the txn was created and is now changed to 8d583756-f36f-4e15-816e-3861b66ed474.
26/08/02 05:27:52 WARN DeltaLog: Change in the table id detected while updating snapshot. 
Previous snapshot = Snapshot(path=s3a://lakehouse/gold/beneficiarios_por_municipio/_delta_log, version=4, metadata=Metadata(8a9eb749-121b-4eb2-8a38-e359138990eb,null,null,Format(parquet,Map()),{"type":"struct","fields":[{"name":"cd_municipio","type":"string","nullable":true,"metadata":{}},{"name":"nm_municipio","type":"string","nullable":true,"metadata":{}},{"name":"qt_beneficiarios_ativos","type":"long","nullable":true,"metadata":{}}]},List(),Map(),Some(1785647531500)), logSegment=LogSegment(s3a://lakehouse/gold/beneficiarios_por_municipio/_delta_log,4,WrappedArray(S3AFileStatu

,namespace,tableName,isTemporary
0,gold,beneficiarios_por_faixa_etaria,False
1,gold,beneficiarios_por_municipio,False
2,gold,beneficiarios_por_operadora,False
3,,raw_csv,False


## Consultas do case
(a) top 5 operadoras — (b) faixa etária com mais beneficiários — (c) beneficiários por município.

In [5]:
run_queries(spark)


(a) Top 5 operadoras por beneficiários ativos:
+------------+------------------------------------------+-----------------------+
|cd_operadora|nm_razao_social                           |qt_beneficiarios_ativos|
+------------+------------------------------------------+-----------------------+
|359017      |NOTRE DAME INTERMÉDICA SAÚDE S.A.         |4324507                |
|301949      |ODONTOPREV S/A                            |2844728                |
|326305      |AMIL ASSISTÊNCIA MÉDICA INTERNACIONAL S.A.|2671950                |
|006246      |SUL AMERICA COMPANHIA DE SEGURO SAÚDE     |2111707                |
|000582      |PORTO SEGURO - SEGURO SAÚDE S/A           |1391580                |
+------------+------------------------------------------+-----------------------+

(b) Faixa etária com mais beneficiários:
+---------------+-----------------------+
|de_faixa_etaria|qt_beneficiarios_ativos|
+---------------+-----------------------+
|40 a 44 anos   |3100823                |
+---